# Reply Mirror -- Multi-Agent Fraud Detection: Run & Inspect

This notebook is the interactive **runner** for the pipeline. All agent/scoring logic lives in the importable `reply_mirror` package at the repo root (`config.py`, `data_loading.py`, `identity.py`, `agents.py`, `memory.py`, `orchestrator.py`, `pipeline.py`, `validator.py`) -- this notebook does not redefine any of it, so there is exactly one implementation to trust.

For a scripted/CI-style run instead of a notebook, the same `reply_mirror.pipeline.run_one` function is wrapped by `run.py` at the repo root (`python run.py --level The_Truman_Show --split train`). Use whichever fits the moment; both call the same code.

**Safety note before you run anything**: only the *first* submission against each level's validation (eval) set counts, and it cannot be undone. Sections 1-4 below only ever touch `train` data. Do not run the validation section (bottom of this notebook) until you actually intend to submit that level for real.

In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "reply_mirror").is_dir() and (candidate / "run.py").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (looked for a reply_mirror/ package and run.py)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

In [ ]:
import pandas as pd

from reply_mirror.config import MAX_LLM_ESCALATIONS, OUTPUT_DIR, USE_AUDIO, USE_LLM, discover_levels
from reply_mirror.memory import MemoryStore
from reply_mirror.pipeline import ValidationSplitGuardError, run_one
from reply_mirror.validator import load_eval_transaction_ids, validate_submission

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## 1. Discover levels

Levels are auto-discovered from whatever folders exist under `dataset/` -- nothing is hardcoded, so levels 4-5 will show up here the moment they're added, with no code changes.

In [ ]:
levels = discover_levels()
for name, lp in sorted(levels.items()):
    train_name = lp.train.name if lp.train else None
    val_name = lp.validation.name if lp.validation else None
    print(f"{name:20s} train={str(train_name):35s} validation={val_name}")

## 2. Configure this run

`USE_LLM` gates the Contextual Reasoning Agent (and, transitively, the Audio Agent -- it only transcribes clips tied to a transaction that's already been escalated to the reasoning agent). Leave it `False` for a fast, free, cheap-signal-only pass; the pipeline degrades gracefully either way. Set it `True` once `.env`'s `OPENAI_API_KEY` is a working OpenRouter key.

In [ ]:
LEVELS_TO_RUN = sorted(levels.keys())  # or e.g. ["The_Truman_Show"]
SPLIT = "train"  # train is safe to iterate on; validation is handled separately at the bottom
USE_LLM_RUN = USE_LLM  # override to True/False here if you want to diverge from .env's default
USE_AUDIO_RUN = USE_AUDIO
LLM_BUDGET = MAX_LLM_ESCALATIONS

print(f"levels={LEVELS_TO_RUN} split={SPLIT!r} use_llm={USE_LLM_RUN} use_audio={USE_AUDIO_RUN} llm_budget={LLM_BUDGET}")

## 3. Run the pipeline

`memory` is a single `MemoryStore` instance shared across every level run in this loop (and persisted to `state/memory_store.json`), so each subsequent level's fusion weights start from a decayed average of everything run before it -- this is the cross-level adaptivity the challenge scores, made visible here rather than hidden inside a CLI invocation.

In [ ]:
memory = MemoryStore.load()
output_paths = {}

for level_name in LEVELS_TO_RUN:
    lp = levels[level_name]
    folder = lp.train if SPLIT == "train" else lp.validation
    if folder is None:
        print(f"Skipping {level_name}: no {SPLIT} folder found on disk.")
        continue
    out_path = run_one(
        level_name,
        SPLIT,
        folder,
        memory,
        use_llm=USE_LLM_RUN,
        use_audio=USE_AUDIO_RUN,
        llm_budget=LLM_BUDGET,
    )
    output_paths[level_name] = out_path

## 4. Inspect results

Visual sanity-check of the highest-scoring flagged transactions per level -- there's no ground-truth label to compute real precision/recall against, so this face-validity read (do the `reasons` actually look like fraud signals, or like an artifact?) is the practical substitute.

In [ ]:
for level_name in output_paths:
    debug_path = OUTPUT_DIR / f"debug_{level_name}_{SPLIT}.csv"
    flagged_ids = set(pd.read_csv(output_paths[level_name], header=None)[0].astype(str))
    debug_df = pd.read_csv(debug_path)

    print(f"\n=== {level_name} [{SPLIT}] -- top 10 flagged by final_score ===")
    cols = ["transaction_id", "amount", "final_score", "behavior_score", "geo_score", "network_score", "comm_score", "economic_score", "reasoning_score"]
    top = debug_df[debug_df["transaction_id"].isin(flagged_ids)].sort_values("final_score", ascending=False).head(10)
    display(top[cols])
    for _, row in top.head(5).iterrows():
        print(f"  {row['transaction_id'][:8]} ({row['final_score']:.3f}): {str(row['reasons'])[:180]}")

### Cross-level memory

What the Memory/Drift Agent has learned so far: fusion weights, threshold quantile, and flag rate recorded per level+split run, plus the accumulated lure-phrase lexicon surfaced by the LLM Contextual Reasoning Agent (empty until a run has `USE_LLM_RUN=True` with a working key).

In [ ]:
memory_rows = []
for key, rec in memory.data["levels"].items():
    row = {"level_split": key, "n_transactions": rec["n_transactions"], "flag_rate": round(rec["flag_rate"], 3)}
    row.update({f"w_{k}": round(v, 3) for k, v in rec["fusion_weights"].items()})
    memory_rows.append(row)
display(pd.DataFrame(memory_rows))
print("Lure lexicon learned so far:", memory.get_lexicon())

## 5. Validate output format

Hard gate per the competition rules: non-empty, not-all, IDs must be a subset of that split's transaction IDs, ASCII, plausible flag rate. `run_one` already runs this automatically and prints the result -- this cell re-runs it standalone so you have an explicit pass/fail summary in one place before submitting anything.

In [ ]:
for level_name, out_path in output_paths.items():
    lp = levels[level_name]
    folder = lp.train if SPLIT == "train" else lp.validation
    result = validate_submission(out_path, load_eval_transaction_ids(folder))
    status = "VALID" if result.ok else "INVALID"
    print(f"{level_name}: {status} -- {result.stats}")
    for err in result.errors:
        print(f"  ERROR: {err}")
    for warn in result.warnings:
        print(f"  WARNING: {warn}")

## 6. Submit against validation -- ONLY when you mean it

This cell is guarded twice on purpose: `run_one` itself refuses to touch a validation split unless `confirm_validation=True` is passed explicitly (see `reply_mirror/pipeline.py`), and this cell additionally requires `CONFIRM_VALIDATION_SUBMIT = True` below to even attempt it. Pick exactly one level per run -- do not loop this over every level at once.

In [ ]:
CONFIRM_VALIDATION_SUBMIT = False  # flip to True only when you are ready to submit LEVEL_TO_SUBMIT for real
LEVEL_TO_SUBMIT = "The_Truman_Show"

if not CONFIRM_VALIDATION_SUBMIT:
    print("CONFIRM_VALIDATION_SUBMIT is False -- not running. Flip it to True when you actually intend to submit.")
else:
    lp = levels[LEVEL_TO_SUBMIT]
    try:
        out_path = run_one(
            LEVEL_TO_SUBMIT,
            "validation",
            lp.validation,
            memory,
            use_llm=USE_LLM_RUN,
            use_audio=USE_AUDIO_RUN,
            llm_budget=LLM_BUDGET,
            confirm_validation=True,
        )
        print(f"Submitted: {out_path}")
    except ValidationSplitGuardError as exc:
        print(f"Blocked: {exc}")